# NODDI Fitting on HCP Data (Subsampled Protocols)

**Purpose:** Fit NODDI model to subsampled HCP DTI protocols using AMICO  
**Data:** Multi-shell acquisition with varying direction counts per shell  
**Protocols:** 20, 30, 45, 60, 75, 90 directions per shell (3 shells, 18 total protocols)  
**Output:** NDI, ODI, FWF maps for each protocol

---

## 1. Setup and Configuration

In [ ]:
#!/usr/bin/env python
"""
NODDI fitting using AMICO on HCP subsampled protocols
Author: Raiyun Kabir
Date: March 31, 2026
"""

import os
import sys
import numpy as np
import nibabel as nib
import amico
from pathlib import Path
import time
import shutil

print("Package versions:")
print(f"  NumPy:   {np.__version__}")
print(f"  NiBabel: {nib.__version__}")
print(f"  AMICO:   {amico.__version__}")

## 2. Configuration Parameters

Define protocol configurations and file paths.

In [ ]:

# ============================================================
# CONFIGURATION - MODIFY THESE
# ============================================================

# Subject and data paths
SUBJECT = "100408"
DATA_ROOT = "/scratch/rkabir5/StarterCodes_Data"
OUTPUT_ROOT = f"{DATA_ROOT}/noddi_results"

# Full (original) DWI data directory
FULL_DATA_DIR = f"{DATA_ROOT}/100408_3T_Diffusion_preproc/{SUBJECT}/T1w/Diffusion"
FULL_DWI  = os.path.join(FULL_DATA_DIR, "data.nii.gz")
MASK_FILE = os.path.join(FULL_DATA_DIR, "nodif_brain_mask.nii.gz")

# Subsampled protocol files directory
# Contains: indices_b{shell}_n{ndir}.txt, bvals_b{shell}_n{ndir}, bvecs_b{shell}_n{ndir}
PROTOCOL_DIR = f"{DATA_ROOT}/subsampled_protocols"

# Shells and direction counts
SHELLS = [1000, 2000, 3000]
NDIRS  = [20, 30, 45, 60, 75, 90]

PROTOCOLS = {}
for shell in SHELLS:
    for ndir in NDIRS:
        protocol_name = f"b{shell}_n{ndir}"
        PROTOCOLS[protocol_name] = {'shell': shell, 'ndir': ndir}

# Get protocol from environment or use all
PROTOCOL_NAME = os.environ.get('PROTOCOL', None)

if PROTOCOL_NAME and PROTOCOL_NAME in PROTOCOLS:
    protocols_to_run = {PROTOCOL_NAME: PROTOCOLS[PROTOCOL_NAME]}
    print(f"Processing single protocol: {PROTOCOL_NAME}")
else:
    protocols_to_run = PROTOCOLS
    if PROTOCOL_NAME:
        print(f"Warning: PROTOCOL={PROTOCOL_NAME} not found. Processing all protocols.")
    else:
        print(f"Processing all {len(PROTOCOLS)} protocols")

# AMICO parameters
B0_THRESHOLD = 50  # Treat b-values < 50 as b0
NUM_THREADS  = 8   # Parallel processing threads

# Get job ID from environment (SLURM_JOB_ID)
JOB_ID   = os.environ.get('SLURM_JOB_ID', 'local')
TEMP_DIR = f"{DATA_ROOT}/temp_noddi_{JOB_ID}"   # temp storage per-protocol

# ============================================================
# Helper functions (defined once outside loop)
# ============================================================

def round_to_shell(bval):
    """Round b-value to nearest standard shell."""
    if bval < 50:
        return 0
    elif bval < 1500:
        return 1000
    elif bval < 2500:
        return 2000
    else:
        return 3000

def create_amico_scheme(bval_file, bvec_file, output_file):
    """Convert FSL bvals/bvecs to AMICO scheme format."""
    bvals = np.loadtxt(bval_file)
    bvecs = np.loadtxt(bvec_file)
    if bvecs.shape[0] == 3:
        bvecs = bvecs.T
    scheme = np.column_stack([bvecs, bvals])
    with open(output_file, 'w') as f:
        f.write("VERSION: BVECTOR\n")
        np.savetxt(f, scheme, fmt='%.6f')
    return scheme

# Initialize AMICO once (expensive — do NOT put inside loop)
print("\nInitializing AMICO framework...")
amico.core.setup()
print("✓ AMICO initialized\n")

print(f"Configuration:")
print(f"  Subject:          {SUBJECT}")
print(f"  Job ID:           {JOB_ID}")
print(f"  Full DWI:         {FULL_DWI}")
print(f"  Protocol dir:     {PROTOCOL_DIR}")
print(f"  Temp dir:         {TEMP_DIR}")
print(f"  Output root:      {OUTPUT_ROOT}")
print(f"  Protocols to run: {list(protocols_to_run.keys())}")
print(f"  Threads:          {NUM_THREADS}")



## 3. Verify Source Files

Check that the full DWI, mask, and all 18 protocol index/bval/bvec files exist before starting.


In [ ]:

# ============================================================
# Verify source files before starting long job
# ============================================================

print("Checking source files...\n")

# Full DWI and mask
for label, fpath in [("Full DWI", FULL_DWI), ("Brain mask", MASK_FILE)]:
    status = "✓" if os.path.exists(fpath) else "✗ MISSING"
    print(f"  {status}  {label}: {fpath}")

# Protocol index / bval / bvec files
print(f"\nChecking protocol files in {PROTOCOL_DIR}:")
missing_protocols = []
for protocol_name, info in PROTOCOLS.items():
    shell, ndir = info['shell'], info['ndir']
    idx_file  = os.path.join(PROTOCOL_DIR, f"indices_b{shell}_n{ndir}.txt")
    bval_file = os.path.join(PROTOCOL_DIR, f"bvals_b{shell}_n{ndir}")
    bvec_file = os.path.join(PROTOCOL_DIR, f"bvecs_b{shell}_n{ndir}")
    ok = all(os.path.exists(f) for f in [idx_file, bval_file, bvec_file])
    status = "✓" if ok else "✗ MISSING"
    print(f"  {status}  {protocol_name}")
    if not ok:
        missing_protocols.append(protocol_name)

if missing_protocols:
    raise FileNotFoundError(f"Missing protocol files for: {missing_protocols}")

if not os.path.exists(FULL_DWI):
    raise FileNotFoundError(f"Full DWI not found: {FULL_DWI}")

print(f"\n✓ All files verified — ready to process {len(protocols_to_run)} protocols")

# ============================================================
# Load full DWI ONCE — all 18 extractions reuse this array
# ============================================================

print(f"\nLoading full DWI (this may take ~30s)...")
t0 = time.time()
full_img      = nib.load(FULL_DWI)
full_dwi_data = full_img.get_fdata()   # shape: (145, 174, 145, 288) ~2 GB
full_affine   = full_img.affine
full_header   = full_img.header
print(f"  ✓ Full DWI loaded: shape={full_dwi_data.shape}, {(time.time()-t0):.1f}s")

# ============================================================
# Main loop — one NODDI fit per protocol
# ============================================================

all_results = {}

for protocol_name, info in protocols_to_run.items():
    shell, ndir = info['shell'], info['ndir']

    print("\n" + "="*70)
    print(f"PROTOCOL: {protocol_name}  (shell={shell}, dirs={ndir})")
    print("="*70)

    # ---- Paths -------------------------------------------------------
    proto_temp_dir = os.path.join(TEMP_DIR, protocol_name)
    os.makedirs(proto_temp_dir, exist_ok=True)

    idx_file  = os.path.join(PROTOCOL_DIR, f"indices_b{shell}_n{ndir}.txt")
    bval_file = os.path.join(PROTOCOL_DIR, f"bvals_b{shell}_n{ndir}")
    bvec_file = os.path.join(PROTOCOL_DIR, f"bvecs_b{shell}_n{ndir}")

    OUTPUT_DIR = Path(OUTPUT_ROOT) / f"noddi_{protocol_name}_{JOB_ID}"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---- Extract volumes (equivalent to fslselectvols) ---------------
    indices = np.loadtxt(idx_file, dtype=int)
    print(f"  Extracting {len(indices)} volumes from full DWI...")
    extracted_data = full_dwi_data[..., indices]   # (145,174,145,n_vols)

    # Save extracted DWI to temp dir
    proto_dwi_path = os.path.join(proto_temp_dir, "data.nii.gz")
    nib.save(nib.Nifti1Image(extracted_data.astype(np.float32), full_affine, full_header),
             proto_dwi_path)
    print(f"  ✓ Saved extracted DWI: {extracted_data.shape}")

    # Copy mask
    proto_mask_path = os.path.join(proto_temp_dir, "nodif_brain_mask.nii.gz")
    shutil.copy2(MASK_FILE, proto_mask_path)

    # ---- Create AMICO scheme from per-protocol bvals/bvecs -----------
    bvals = np.loadtxt(bval_file)
    bvals_rounded = np.array([round_to_shell(b) for b in bvals])
    bval_rounded_path = os.path.join(proto_temp_dir, "bvals_rounded")
    np.savetxt(bval_rounded_path, bvals_rounded, fmt='%.1f', newline=' ')

    scheme_file = os.path.join(proto_temp_dir, "NODDI.scheme")
    scheme = create_amico_scheme(bval_rounded_path, bvec_file, scheme_file)
    print(f"  ✓ AMICO scheme created: {len(scheme)} volumes")

    # ---- AMICO fitting ----------------------------------------------
    print("  Setting up AMICO evaluation...")
    ae = amico.Evaluation(proto_temp_dir, ".")
    ae.load_data(
        dwi_filename="data.nii.gz",
        scheme_filename="NODDI.scheme",
        mask_filename="nodif_brain_mask.nii.gz",
        b0_thr=B0_THRESHOLD
    )
    ae.set_model("NODDI")

    print("  Generating kernels...")
    ae.generate_kernels(regenerate=True)
    ae.load_kernels()

    print("  Fitting NODDI model...")
    t_fit = time.time()
    ae.fit()
    elapsed = time.time() - t_fit
    print(f"  ✓ Fitting done in {elapsed/60:.1f} min")

    ae.save_results()

    # ---- Copy results to final output dir ---------------------------
    amico_results_dir = os.path.join(proto_temp_dir, "AMICO", "NODDI")
    noddi_files = ["fit_NDI.nii.gz", "fit_ODI.nii.gz", "fit_FWF.nii.gz",
                   "fit_dir.nii.gz", "config.pickle"]

    print(f"  Copying results to {OUTPUT_DIR}")
    for fname in noddi_files:
        src = os.path.join(amico_results_dir, fname)
        dst = OUTPUT_DIR / fname
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"    ✓ {fname} ({os.path.getsize(src)/1e6:.1f} MB)")
        else:
            print(f"    ✗ {fname} NOT FOUND")

    # Save protocol metadata
    with open(OUTPUT_DIR / "protocol_info.txt", "w") as f:
        f.write(f"protocol: {protocol_name}\nshell: {shell}\ndirs: {ndir}\n"
                f"n_volumes: {len(indices)}\nfit_time_min: {elapsed/60:.1f}\njob_id: {JOB_ID}\n")

    # ---- Cleanup temp dir to save disk space -------------------------
    shutil.rmtree(proto_temp_dir)
    print(f"  ✓ Temp dir removed")

    # ---- Statistics --------------------------------------------------
    ndi = nib.load(OUTPUT_DIR / "fit_NDI.nii.gz").get_fdata()
    odi = nib.load(OUTPUT_DIR / "fit_ODI.nii.gz").get_fdata()
    fwf = nib.load(OUTPUT_DIR / "fit_FWF.nii.gz").get_fdata()
    mask_data = nib.load(MASK_FILE).get_fdata() > 0

    stats = {}
    for name, data in [("NDI", ndi), ("ODI", odi), ("FWF", fwf)]:
        vals = data[mask_data]
        vals = vals[np.isfinite(vals) & (vals > 0)]
        stats[name] = {'mean': vals.mean(), 'std': vals.std()}

    all_results[protocol_name] = {
        'shell': shell, 'ndir': ndir,
        'n_volumes': len(indices),
        'fitting_time_min': elapsed / 60,
        'stats': stats
    }

    print(f"\n  {protocol_name} — NDI: {stats['NDI']['mean']:.4f} | "
          f"ODI: {stats['ODI']['mean']:.4f} | FWF: {stats['FWF']['mean']:.4f}")
    print(f"  ✓ {protocol_name} COMPLETE\n")



## 4. Summary of All Protocols


In [ ]:

print("\n" + "="*70)
print("NODDI SUBSAMPLING STUDY COMPLETE")
print("="*70)
print(f"\nJob ID:    {JOB_ID}")
print(f"Subject:   {SUBJECT}")
print(f"Protocols: {len(all_results)}/{len(protocols_to_run)}")

print(f"\n{'Protocol':<15s} {'Shell':>6s} {'Dirs':>5s} {'Vols':>5s} {'Time(m)':>8s} {'NDI':>8s} {'ODI':>8s} {'FWF':>8s}")
print("-" * 67)

for protocol_name in sorted(all_results.keys()):
    r = all_results[protocol_name]
    s = r['stats']
    print(f"{protocol_name:<15s} {r['shell']:6d} {r['ndir']:5d} {r['n_volumes']:5d} "
          f"{r['fitting_time_min']:8.2f} {s['NDI']['mean']:8.4f} {s['ODI']['mean']:8.4f} {s['FWF']['mean']:8.4f}")

print("="*70)
print(f"Output: {OUTPUT_ROOT}/noddi_{{protocol}}_{JOB_ID}/")
print("="*70)
